# Cross-Model UQ Importance Evaluation

Compares trained policies head-to-head to isolate the contribution of UQ to learned behavior.
Unlike `eval_patrolling_uq.ipynb` (test-time UQ intervention on a single model), this notebook
evaluates *different* trained models at the same eval settings — answering "does training with
UQ change what the policy learns?" rather than "does this policy react to the UQ feature?".

Models (all trained via `ptrain_patrolling.py`):
- **Comms+Pred (no UQ)** — `train_patrolling_comms_pred`, baseline without UQ signal
- **Comms+Pred+UQ (merge_min)** — `train_patrolling_comms_pred_uq`, UQ on receive side
- **Comms+Pred+UQ-Gated (T=1.0)** — `neighbors_with_comm_uq_threshold`, temperature 1.0
- **Comms+Pred+UQ-Gated (T=0.5)** — temperature 0.5
- **Comms+Pred+UQ-Gated (T=0.1)** — temperature 0.1 (sharpest gating)

Sweeps observation radius with many eval episodes per (model, radius) for noise control.
Each episode resets the graph and spawn config, so episodes are independent draws.

## Configuration

In [1]:
from datetime import datetime
datetime_str = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = "./results/patrolling_uq_crossmodel_" + datetime_str + ".zarr"

# Pin output file once eval has been run, so re-running the notebook just loads results.
# output_file = "./results/patrolling_uq_crossmodel_<pinned>.zarr"

# Currently in paper:
# output_file = "/data/anthony/dev/qmas_task_allocation/scripts/final/results/patrolling_uq_crossmodel_20260518_003802.zarr"

## Setup

In [2]:
%reload_ext autoreload
%autoreload 2
from onpolicy.scripts.eval.eval_pettingzoo import main
import os
import time
from multiprocessing import Process
from args import *

MAX_PROCESSES = 200
GPUS = [0, 1, 2, 3, 4, 5, 6, 7]
GPU_AUTO_SELECT = True

def get_gpu_least_used():
    gpu_memory = os.popen("nvidia-smi --query-gpu=memory.used --format=csv").read()
    gpu_memory = gpu_memory.split("\n")[1:-1]
    gpu_memory = [int(x.split()[0]) for x in gpu_memory]
    gpu_memory = [gpu_memory[i] if i in GPUS else float('inf') for i in range(len(gpu_memory))]
    least_used_gpu = gpu_memory.index(min(gpu_memory))
    return least_used_gpu

# Default eval args. Bumped eval_episodes well above the original 3 — since each episode
# resets the graph + spawn config, episodes are independent and a single run with many
# episodes gives the same statistical power as a multi-seed sweep.
default_args = {}
default_args["eval_output_file"] = wrap_arg_val(output_file)
default_args["cuda"] = wrap_arg_val(True)
default_args["cuda_idx"] = wrap_arg_val(0)
default_args["cuda_idx_predictor"] = default_args["cuda_idx"]
default_args["eval_episodes"] = wrap_arg_val(10)
default_args["episode_length"] = wrap_arg_val(150)
default_args["seed"] = wrap_arg_val(2)

default_args["observation_mask"] = wrap_arg_val(True)
default_args["graph_random"] = wrap_arg_val(True)

default_args["num_agents"] = wrap_arg_val(8)

# Fix the obs-radius randomization range so the swept observation_radius is what's actually used.
default_args["observation_radius_random_min"] = wrap_arg_val(0.0)
default_args["observation_radius_random_max"] = wrap_arg_val(0.0)

# Match training-time autoregression depth for the UQ-aware comms models.
default_args["diffusion_autoregression_steps"] = wrap_arg_val(7)

## Model Paths

Fill in the wandb run directories for each trained model. Each entry maps a human-readable
series name to the `files/` directory of the corresponding wandb run.

Leave any entry as `None` to skip that model.

In [3]:
MODELS = {
    # Baseline: comms + prediction, NO UQ signal in policy or comms gating.
    "Observations (Boolean)":
        "/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-draft1/wandb/run-20260521_023023-9sgybv9e/files",

    # UQ on receive side (merge_minimum_uq), boolean comm action.
    "Predictions+UQ (Boolean)":
        "/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260521_023023-7pgsd7xf/files",

    # UQ-threshold-gated comms: actor emits a comm_threshold, env broadcasts iff
    # Bernoulli(sigmoid((u - threshold)/T)) fires. Three temperatures from soft to sharp.
    "Predictions+UQ (Threshold, T=1.0)":
        "/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260521_023023-eozh4t4o/files",
    "Predictions+UQ (Threshold, T=0.5)":
        "/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260521_023023-qd2nyd6h/files",
    "Predictions+UQ (Threshold, T=0.1)":
        "/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260521_023023-muk12tx1/files",
}

MODELS = {name: path for name, path in MODELS.items() if path is not None}
print(f"{len(MODELS)} model(s) configured:")
for name in MODELS:
    print(f"  - {name}")

5 model(s) configured:
  - Observations (Boolean)
  - Predictions+UQ (Boolean)
  - Predictions+UQ (Threshold, T=1.0)
  - Predictions+UQ (Threshold, T=0.5)
  - Predictions+UQ (Threshold, T=0.1)


## Evaluation

In [4]:
import numpy as np

obs_radii = np.arange(5.0, 51.0, 5.0).tolist()

if os.path.exists(output_file):
    print("Output file already exists. Skipping evaluation.")
else:
    process_args = []

    def add_process(args):
        process_args.append(args)

    def start_process(args):
        p = Process(target=main, args=(args,))
        p.start()
        return p

    for series_name, model_dir in MODELS.items():
        for obs_radius in obs_radii:
            args = load_args(model_dir, default_args)
            args["eval_series"] = series_name
            args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
            args["observation_radius"] = wrap_arg_val(obs_radius)
            add_process(args)

    print(f"Queued {len(process_args)} eval runs across {len(MODELS)} model(s), "
          f"{len(obs_radii)} obs radii.")

    running_processes = []
    time_start = time.time()
    failed_processes = []

    while len(process_args) > 0 or len(running_processes) > 0:
        while len(running_processes) < MAX_PROCESSES and len(process_args) > 0:
            args = process_args.pop(0)
            if GPU_AUTO_SELECT:
                least_used_gpu = get_gpu_least_used()
                args["cuda_idx"] = wrap_arg_val(least_used_gpu)
                args["cuda_idx_predictor"] = wrap_arg_val(least_used_gpu)
            p = start_process(convert_args(args))
            running_processes.append(p)

        for p in running_processes:
            if not p.is_alive():
                p.join()
                print(f"Process {p.pid} finished with exit code {p.exitcode}. "
                      f"Running: {len(running_processes)-1}, queued: {len(process_args)}, "
                      f"failed: {len(failed_processes)}")
                if p.exitcode != 0:
                    failed_processes.append(p)
        running_processes = [p for p in running_processes if p.is_alive()]

    time_end = time.time()
    print(f"Total evaluation time: {time_end - time_start:.2f} seconds")

    if failed_processes:
        raise RuntimeError(f"{len(failed_processes)} subprocess(es) failed with non-zero exit codes")

    print("All processes finished successfully!")

Queued 50 eval runs across 5 model(s), 10 obs radii.
Converted arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 49, 55, 63, 74, 77, 103, 105], '3': [2, 13, 16, 23, 55, 61], '4': '3.10.16', '5': '0.19.11', '8': [5, 9], '12': '0.19.11', '13': 'linux-x86_64'}}}", '--accept_ratio', '0.5', '--action_full_max_nodes', '40', '--action_method', 'neighbors_with_comm_boolean', '--action_neighbors_max_degree', '10', '--action_velocity_acceptance_radius', '5', '--agent_speed', '40', '--algorithm_class', 'qmas.variant_two_modules.new.algorithm.QmasAlgorithm', '--algorithm_name', 'mappo', '--alpha', '1', '--attrition_fixed_times', '[]', '--attrition_method', 'none', '--attrition_min_agents', '2', '--attrition_random_probability', '0', '--beta', '0.1', '--broadcast_uncertainty_threshold', '0.5', '--clip_param', '0.2', '--comm_uq_threshold_temperature', '1', '--communication_model', 'none', '--communication_

/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:162: UserWarning: Unknown arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 49, 55, 63, 74, 77, 103, 105], '3': [2, 13, 16, 23, 55, 61], '4': '3.10.16', '5': '0.19.11', '8': [5, 9], '12': '0.19.11', '13': 'linux-x86_64'}}}", '--episode_max', '101', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")


u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be False
Converted arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 49, 55, 63, 74, 77, 103, 105], '3': [2, 13, 16, 23, 55, 61], '4': '3.10.16', '5': '0.19.11', '8': [5, 9], '12': '0.19.11', '13': 'linux-x86_64'}}}", '--accept_ratio', '0.5', '--action_full_max_nodes', '40', '--action_method', 'neighbors_with_comm_boolean', '--action_neighbors_max_degree', '10', '--action_velocity_acceptance_radius', '5', '--agent_speed', '40', '--algorithm_class', 'qmas.variant_two_modules.new.algorithm.QmasAlgorithm', '--algorithm_name', 'mappo', '--alpha', '1', '--attrition_fixed_times', '[]', '--attrition_method', 'none', '--attrition_min_agents', '2', '--attrition_random_probability', '0', '--beta', '0.1', '--broadcast_uncertainty_threshold', '0.5', '--clip_param', '0.2', '--comm_uq_threshold_temperature', '1', '--co

/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:162: UserWarning: Unknown arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 49, 55, 63, 74, 77, 103, 105], '3': [2, 13, 16, 23, 55, 61], '4': '3.10.16', '5': '0.19.11', '8': [5, 9], '12': '0.19.11', '13': 'linux-x86_64'}}}", '--episode_max', '101', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")


u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be False
Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments.


...

Restored models from /data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260521_023023-muk12tx1/files
Restored models from /data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260521_023023-muk12tx1/filesInitialized QmasAlgorithm with 3 predictors. Threaded training: True

Buffer initialized with episode length 150
Initialized QmasAlgorithm with 3 predictors. Threaded training: True
Buffer initialized with episode length 150
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 9182.
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 9182.
R_Actor: Use GNN: True, Use CNN: False, Use MLP: False, Use RNN: False, Use Encoder: False
Action output distributions:
  0: DiagGaussian, action_dim: 1, available_action_dim: inf
  1: Categorical, action_dim: 1, available_action_dim: 10
QmasPolicy: Using di

/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 9182.
Restored models from /data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260521_023023-muk12tx1/files
Initialized QmasAlgorithm with 3 predictors. Threaded training: True
Restored models from /data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260521_023023-muk12tx1/files
Initialized QmasAlgorithm with 3 predictors. Threaded training: True
Buffer initialized with episode length 150
Buffer initialized with episode length 150
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 9182.
R_Actor: Use GNN: True, Use CNN: False, Use MLP: False, Use RNN: False, Use Encoder: False
Action output distributions:
  0: DiagGaussian, action_dim: 1, available_action_dim: inf
  1: Categorical, action_dim: 1, available_action_dim: 10
R_Critic: Use CNN: F

/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 9182.
Restored models from /data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260521_023023-muk12tx1/files
Initialized QmasAlgorithm with 3 predictors. Threaded training: True
Buffer initialized with episode length 150
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 9182.
Restored models from /data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260521_023023-muk12tx1/files
Initialized QmasAlgorithm with 3 predictors. Threaded training: True
Buffer initialized with episode length 150
Process 278922 finished with exit code 0. Running: 49, queued: 0, failed: 0
Process 279726 finished with exit code 0. Running: 47, queued: 0, failed: 0
Process 278827 finished with exit code 0. Running: 46, queued: 0, failed: 0
Process 282363 finished with e

## Analysis

In [5]:
import zarr
output = zarr.open(output_file, mode='r')
print(output.tree())
print(f"Output file: {os.path.abspath(output_file)}")
print(f"To pin results, set in first cell: output_file = '{os.path.abspath(output_file)}'")

/
 ├── Observations (Boolean)
 │   ├── obs_radius_10.0
 │   │   ├── agent_count (10, 150) float32
 │   │   ├── avg_idleness (10, 150) float32
 │   │   ├── communication
 │   │   │   └── requests_made (10, 150) float32
 │   │   ├── deltaSteps (10, 150, 8, 1) float32
 │   │   ├── node_visits
 │   │   │   ├── node_0 (10, 150) float32
 │   │   │   ├── node_1 (10, 150) float32
 │   │   │   ├── node_10 (10, 150) float32
 │   │   │   ├── node_11 (10, 150) float32
 │   │   │   ├── node_12 (10, 150) float32
 │   │   │   ├── node_13 (10, 150) float32
 │   │   │   ├── node_14 (10, 150) float32
 │   │   │   ├── node_15 (10, 150) float32
 │   │   │   ├── node_16 (10, 150) float32
 │   │   │   ├── node_17 (10, 150) float32
 │   │   │   ├── node_18 (10, 150) float32
 │   │   │   ├── node_19 (10, 150) float32
 │   │   │   ├── node_2 (10, 150) float32
 │   │   │   ├── node_20 (10, 150) float32
 │   │   │   ├── node_21 (10, 150) float32
 │   │   │   ├── node_22 (10, 150) float32
 │   │   │   ├── node_23

In [6]:
import numpy as np

def episode_requests(data):
    """Total communication requests per episode (1-D array of length n_episodes)."""
    r = np.array(data['communication']['requests_made'][...])
    return r.reshape(r.shape[0], -1).sum(axis=1)

def episode_idleness(data):
    """Time-averaged avg. idleness per episode (1-D array of length n_episodes)."""
    arr = np.array(data['avg_idleness'][...])
    # Reduce all non-episode axes to get one scalar per episode.
    if arr.ndim > 1:
        arr = arr.reshape(arr.shape[0], -1).mean(axis=1)
    return arr

# For each (series, obs_radius), pull per-episode arrays and compute mean + SEM.
summary = {}
for series in output:
    radii_data = {}
    for exp in output[series]:
        exp_data = output[series][exp]
        obs_r = float(exp_data.attrs.get('observation_radius', 0.0))
        radii_data[obs_r] = {
            'idleness': episode_idleness(exp_data),
            'requests': episode_requests(exp_data),
        }
    radii = sorted(radii_data.keys())
    summary[series] = {
        'obs_radius':     radii,
        'idleness_mean':  [float(radii_data[r]['idleness'].mean()) for r in radii],
        'idleness_sem':   [float(radii_data[r]['idleness'].std(ddof=1) / np.sqrt(len(radii_data[r]['idleness']))) for r in radii],
        'requests_mean':  [float(radii_data[r]['requests'].mean()) for r in radii],
        'requests_sem':   [float(radii_data[r]['requests'].std(ddof=1) / np.sqrt(len(radii_data[r]['requests']))) for r in radii],
        'n_episodes':     [int(len(radii_data[r]['idleness'])) for r in radii],
    }
    print(f"{series}: {len(radii)} obs radii, n_episodes per point: {summary[series]['n_episodes']}")

Observations (Boolean): 10 obs radii, n_episodes per point: [10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
Predictions+UQ (Boolean): 10 obs radii, n_episodes per point: [10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
Predictions+UQ (Threshold, T=0.1): 10 obs radii, n_episodes per point: [10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
Predictions+UQ (Threshold, T=0.5): 10 obs radii, n_episodes per point: [10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
Predictions+UQ (Threshold, T=1.0): 10 obs radii, n_episodes per point: [10, 10, 10, 10, 10, 10, 10, 10, 10, 10]


### Idleness vs. Observation Radius

In [7]:
import plotly.graph_objects as go
import plotly.express as px

def _hex_to_rgba(hex_color, alpha=0.2):
    c = hex_color.lstrip('#')
    r, g, b = int(c[0:2], 16), int(c[2:4], 16), int(c[4:6], 16)
    return f'rgba({r},{g},{b},{alpha})'

series_order = list(summary.keys())
series_colors = {s: c for s, c in zip(series_order, px.colors.qualitative.Plotly)}

fig = go.Figure()
for series in series_order:
    color = series_colors[series]
    s = summary[series]
    xs = np.array(s['obs_radius'])
    ys = np.array(s['idleness_mean'])
    es = np.array(s['idleness_sem'])

    fig.add_trace(go.Scatter(
        x=np.concatenate([xs, xs[::-1]]),
        y=np.concatenate([ys + es, (ys - es)[::-1]]),
        fill='toself',
        fillcolor=_hex_to_rgba(color, alpha=0.15),
        line=dict(color='rgba(255,255,255,0)'),
        legendgroup=series, showlegend=False, hoverinfo='skip',
    ))
    fig.add_trace(go.Scatter(
        x=xs, y=ys, mode='lines+markers',
        name=series, line=dict(color=color), marker=dict(color=color, size=8),
        legendgroup=series,
        hovertemplate=f"{series}<br>obs_radius=%{{x:.1f}}<br>idleness=%{{y:.4f}}<extra></extra>",
    ))

fig.update_layout(
    title="Time-Averaged Avg. Idleness vs. Observation Radius (shaded: ±1 SEM across episodes)",
    xaxis_title="Observation Radius",
    yaxis_title="Time-Averaged Avg. Idleness (s)",
    height=600, width=900, hovermode='closest', autosize=True,
    margin=dict(l=60, r=20, t=60, b=120),
    legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.5),
)
fig.show()

### Communication Rate vs. Observation Radius

In [8]:
fig = go.Figure()
for series in series_order:
    color = series_colors[series]
    s = summary[series]
    xs = np.array(s['obs_radius'])
    ys = np.array(s['requests_mean'])
    es = np.array(s['requests_sem'])

    fig.add_trace(go.Scatter(
        x=np.concatenate([xs, xs[::-1]]),
        y=np.concatenate([ys + es, (ys - es)[::-1]]),
        fill='toself',
        fillcolor=_hex_to_rgba(color, alpha=0.15),
        line=dict(color='rgba(255,255,255,0)'),
        legendgroup=series, showlegend=False, hoverinfo='skip',
    ))
    fig.add_trace(go.Scatter(
        x=xs, y=ys, mode='lines+markers',
        name=series, line=dict(color=color), marker=dict(color=color, size=8),
        legendgroup=series,
        hovertemplate=f"{series}<br>obs_radius=%{{x:.1f}}<br>requests=%{{y:.1f}}<extra></extra>",
    ))

fig.update_layout(
    title="Total Communication Requests vs. Observation Radius (shaded: ±1 SEM across episodes)",
    xaxis_title="Observation Radius",
    yaxis_title="Total Communication Requests (per episode)",
    height=600, width=900, hovermode='closest', autosize=True,
    margin=dict(l=60, r=20, t=60, b=120),
    legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.5),
)
fig.show()

### Pareto: Idleness vs. Communication Rate

Each model traces a curve through (comm rate, idleness) space as observation radius varies.
A model that achieves lower idleness at lower comm cost dominates.

In [9]:
fig = go.Figure()
markers = ['circle', 'square', 'diamond', 'cross', 'x']
for series in series_order:
    color = series_colors[series]
    s = summary[series]
    xs = np.array(s['requests_mean'])
    ys = np.array(s['idleness_mean'])
    radii = s['obs_radius']

    fig.add_trace(go.Scatter(
        x=xs, y=ys, mode='markers+text',
        name=series, line=dict(color=color), marker=dict(color=color, size=8, symbol=markers[series_order.index(series) % len(markers)]),
        legendgroup=series,
        # text=[f"{r:.0f}" for r in radii],
        textposition='top center',
        textfont=dict(color=color, size=9),
        hovertemplate=[
            f"{series}<br>obs_radius={r:.1f}<br>requests=%{{x:.1f}}<br>idleness=%{{y:.4f}}<extra></extra>"
            for r in radii
        ],
    ))

fig.update_layout(
    title="Idleness vs. Communication",
    xaxis_title="Total Communication Requests (per episode)",
    yaxis_title="Time-Averaged Avg. Idleness (s)",
    height=600, width=900, hovermode='closest', autosize=True,
    margin=dict(l=60, r=20, t=60, b=120),
    legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.5),
)
fig.show()


### Summary Table

In [10]:
import pandas as pd
rows = []
for series in series_order:
    s = summary[series]
    for i, r in enumerate(s['obs_radius']):
        rows.append({
            'series': series,
            'obs_radius': r,
            'idleness_mean': s['idleness_mean'][i],
            'idleness_sem':  s['idleness_sem'][i],
            'requests_mean': s['requests_mean'][i],
            'requests_sem':  s['requests_sem'][i],
            'n_episodes':    s['n_episodes'][i],
        })
df = pd.DataFrame(rows)
df

ModuleNotFoundError: No module named 'pandas'